# SignSense AI — MobileNetV3 CNN Training (Kaggle)

**Model:** MobileNetV3Small transfer learning on 224×224 hand crop images  
**Target accuracy:** > 90% | **Runtime:** ~2h on Kaggle T4 GPU

### Before you start
1. Settings → Accelerator → **GPU T4 x2**
2. Add dataset: **+ Add Data** → search `grassknoted/asl-alphabet` → Add
3. Run `kaggle_train_mlp.ipynb` first (it generates the image crops)
4. Run all cells top to bottom

### Two-phase training
- Phase 1 (~20 epochs): Frozen base, train head only
- Phase 2 (~40 epochs): Unfreeze top layers, fine-tune with low LR

In [ ]:
# ── Cell 1: Setup paths ───────────────────────────────────────────────────────
import os

WORKING_DIR   = '/kaggle/working'
MODELS_DIR    = f'{WORKING_DIR}/models'
LOGS_DIR      = f'{WORKING_DIR}/logs/cnn'
KAGGLE_INPUT  = '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

print(f'Models will be saved to: {MODELS_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# NOTE: Kaggle requires internet to be enabled in Settings (right sidebar)
import subprocess, sys

try:
    import mediapipe as mp
    print(f'✅ mediapipe already installed: {mp.__version__}')
except ImportError:
    print('Installing mediapipe (requires internet enabled in Settings)...')
    !pip install -q 'protobuf>=5.28.0' 'mediapipe>=0.10.18'
    import mediapipe as mp
    print(f'✅ mediapipe installed: {mp.__version__}')

!pip install -q scikit-learn tqdm
print('✅ All dependencies ready.')

In [ ]:
# ── Cell 3: Clone project repo from GitHub ────────────────────────────────────
import os, sys

REPO_URL = 'https://github.com/prateek1756/sign-language-detection.git'

if not os.path.exists(f'{WORKING_DIR}/sign-language-detection'):
    !git clone {REPO_URL} {WORKING_DIR}/sign-language-detection
else:
    !git -C {WORKING_DIR}/sign-language-detection pull

BACKEND_PATH = f'{WORKING_DIR}/sign-language-detection/backend'
sys.path.insert(0, BACKEND_PATH)

from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'✅ Repo ready — {NUM_CLASSES} classes')

In [ ]:
# ── Cell 4: Generate or verify image crops ────────────────────────────────────
import os, shutil
from pathlib import Path

RAW_ASL_DIR   = f'{BACKEND_PATH}/data/raw/ASL'
PROCESSED_DIR = f'{BACKEND_PATH}/data/processed/ASL'
CROPS_DIR     = f'{PROCESSED_DIR}/crops'

if os.path.exists(CROPS_DIR):
    total = sum(len(list(d.glob('*.jpg'))) for d in Path(CROPS_DIR).iterdir() if d.is_dir())
    print(f'✅ Image crops found: {total:,} total')
else:
    if not os.path.exists(KAGGLE_INPUT):
        raise FileNotFoundError('Add grassknoted/asl-alphabet dataset first.')

    print('Copying dataset and generating crops (~15-20 min)...')
    os.makedirs(RAW_ASL_DIR, exist_ok=True)
    for class_dir in Path(KAGGLE_INPUT).iterdir():
        if class_dir.is_dir():
            dest = Path(RAW_ASL_DIR) / class_dir.name
            if not dest.exists():
                shutil.copytree(str(class_dir), str(dest))

    !python {BACKEND_PATH}/src/preprocess.py --all --augment --aug_factor 3

    if not os.path.exists(CROPS_DIR):
        raise RuntimeError('Crop generation failed. Check preprocess.py output.')

    total = sum(len(list(d.glob('*.jpg'))) for d in Path(CROPS_DIR).iterdir() if d.is_dir())
    print(f'✅ Generated {total:,} image crops')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('✅ GPU memory growth enabled.')
else:
    print('WARNING: No GPU. CNN training will take 8-12h. Not recommended.')

In [ ]:
# ── Cell 6: Train MobileNetV3 (Phase 1 + Phase 2) ────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'training_config', 'configs']):
        del sys.modules[mod]

from configs.training_config import CNNConfig
from src.train import train_cnn

cfg = CNNConfig()
cfg.save_dir = Path(MODELS_DIR)
cfg.log_dir  = Path(LOGS_DIR)
cfg.mixed_precision = True

print(f'  base_model:     {cfg.base_model}')
print(f'  Phase 1 epochs: {cfg.phase1_epochs}  lr={cfg.phase1_lr}')
print(f'  Phase 2 epochs: {cfg.phase2_epochs}  lr={cfg.phase2_lr}')
print()

model = train_cnn(cfg)
if model is None:
    print('ERROR: CNN training failed. Check that image crops exist.')
else:
    print('\n✅ Training complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(MODELS_DIR)

results = ev.evaluate('asl_mobilenet', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: Verify saved model ────────────────────────────────────────────────
import os, numpy as np, tensorflow as tf

print('Files saved:')
for f in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

loaded = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'asl_mobilenet.keras'))
dummy  = np.zeros((1, 224, 224, 3), dtype=np.float32)
pred   = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f}')
print('\n✅ Model verified. Download asl_mobilenet.keras from the Output tab.')

In [ ]:
# ── Cell 9: Benchmark all models (run after all 3 notebooks complete) ─────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(MODELS_DIR)
ev.benchmark_all(split='test')